<a href="https://colab.research.google.com/github/KutyshenkoYana/ITStep-AI/blob/superwised_learning/SL_HomeWork_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/HalyshAnton/IT-Step-Pyton-AI/main/module3/data/House_Rent_Dataset.csv", index_col="Unnamed: 0")
df = df.drop(columns="Posted On")

[Ігформація про дані](https://www.kaggle.com/datasets/iamsouravbanerjee/house-rent-prediction-dataset)

In [3]:
df.head()

,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2.0,10000.0,1100.0,Ground out of 2,NaN,NaN,Kolkata,Unfurnished,Bachelors/Family,2.0,Contact Owner
1,2.0,20000.0,800.0,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1.0,NaN
2,2.0,17000.0,1000.0,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1.0,Contact Owner
3,2.0,10000.0,800.0,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1.0,Contact Owner
4,2.0,7500.0,850.0,1 out of 2,Carpet Area,NaN,Kolkata,Unfurnished,Bachelors,1.0,Contact Owner


# Завдання 1
* Очистіть дані від викидів
* Отримайте ознаки та цільову змінну(X, y)
* Розділіть дані на тренувальні та тестові

In [4]:
from sklearn.model_selection import train_test_split

numeric_columns = ["BHK", "Size", "Bathroom"]
categorical_columns = ["Floor", "Area Type", "Area Locality", "City", "Furnishing Status", "Tenant Preferred", "Point of Contact"]

Q1 = df["Rent"].quantile(0.25)
Q3 = df["Rent"].quantile(0.75)
IQR = Q3 - Q1

df = df[
    (df["Rent"] >= Q1 - 1.5 * IQR) &
    (df["Rent"] <= Q3 + 1.5 * IQR)
]

X = df.drop(columns=["Rent"])
y = df["Rent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Завдання 2
Створіть Pipeline для обробки даних числових та категоріальних даних

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

# Завдання 3
Об'єднайте препроцесор з деревом рішень в один пайплайн

In [6]:
from sklearn.tree import DecisionTreeRegressor

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['BHK', 'Size', 'Bathroom']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Floor', 'Area Type',
                                                   'Area Locality', 'City',
                                                   'Furnishing Status',
                                                   'Tenant Preferred',
                                                   'Point of Contact'])])),
                ('model', DecisionTreeRegressor(random_state=42))])

# Завдання 4
Виведіть основні метрики моделі

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

MAE: 5733.894736842105
RMSE: 8587.080110947823
R2: 0.6503278247031077


# Завдання 5
Змініть параметри моделі та попробуйте покращити результат та натренуйте його

In [8]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    ))
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['BHK', 'Size', 'Bathroom']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Floor', 'Area Type',
                                                   'Area Locality', 'City',
                                                   'Furnishing Status',
                                                   'Tenant Preferred',
                                                   'Point of Contact'])])),
                ('model',
                 DecisionTreeRegressor(max_depth=10, min_samples_leaf=2,
                                       min_samples_split=5, random_state=42))])

# Завдання 6
Спрогнозуйте ціну оренди на тестових даних та порівняйте зі справжніми

In [9]:
y_pred = model.predict(X_test)

result = X_test.copy()
result["Real Rent"] = y_test
result["Predicted Rent"] = y_pred

print(result[["Real Rent", "Predicted Rent"]])

      Real Rent  Predicted Rent
4511     7000.0    12791.138249
1265    41000.0    35375.000000
339      7000.0     9708.219178
2688    30000.0    47500.000000
1974    30000.0    30636.363636
...         ...             ...
3700     5000.0     7867.708333
3030    14000.0    16152.475248
1062    50000.0    35818.181818
263     15000.0    13900.000000
3942    23000.0    13244.274809

[779 rows x 2 columns]
